# Phase 8 — IMU-Guided Frame Selection

## Question
Can IMU gyroscope data reduce VGGT's compute cost by selecting only
informative frames — without sacrificing pose accuracy?

## Motivation
Uniform time-based sampling wastes compute on near-duplicate frames during
slow/static motion.  The IMU gyroscope measures cumulative rotation
continuously at 200 Hz (essentially free).  We use it to gate frame
selection: accept a new frame only when the camera has rotated ≥ θ° since
the last accepted frame.  This guarantees every frame fed to VGGT
represents a meaningfully distinct viewpoint.

## Experiments
1. **Selection visualisation** — which frames are picked at θ ∈ {5,8,12,18}°?
2. **Accuracy vs N** — ATE for uniform vs IMU-guided selection at 518 px
   (matched frame counts).
3. **Combined savings** — IMU-guided + 224 px vs uniform + 518 px:
   can we match accuracy at 6× lower compute?
4. **Summary table** — speedup, VRAM saving, ATE change.

## 0 — Setup

In [ ]:
import subprocess, sys, os

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *args])

def git(*args):
    subprocess.check_call(["git"] + list(args))

pip("--upgrade", "numpy")
pip("plotly", "pandas", "pillow", "scipy", "matplotlib")

if not os.path.isdir("vggt"):
    git("clone", "--depth", "1", "https://github.com/facebookresearch/vggt.git")
    pip("-e", "vggt")

if not os.path.isdir("vggt-eval"):
    git("clone", "--depth", "1", "https://github.com/insha-py/vggt-eval.git")
else:
    git("-C", "vggt-eval", "pull", "--ff-only")

sys.path.insert(0, "vggt-eval")
sys.path.insert(0, "vggt")

print("Packages installed. Restarting kernel ...")
try:
    import IPython
    IPython.Application.instance().kernel.do_shutdown(True)
except Exception:
    print("Restart the kernel manually, then run from the Imports cell.")

## 1 — Imports

In [ ]:
import os, sys, gc, time

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

for p in ["vggt-eval", "vggt"]:
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import pandas as pd
import torch
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from src.tum_vi   import TUMVIDataset
from src.imu      import IMUPreintegrator, estimate_gyro_bias, select_frames_by_rotation
from src.pipeline import VGGTPipeline, load_images_from_list, run_vggt_inference
from src.metrics  import compute_ate, compute_rpe

np.random.seed(42)
torch.manual_seed(42)

# Hard cap: max frames per VGGT run — safe at 518 px on T4 16 GB
# Defined here so it is always available regardless of cell execution order.
MAX_VGGT_FRAMES = 8

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device          : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"MAX_VGGT_FRAMES : {MAX_VGGT_FRAMES}")

## 2 — Download Data

In [ ]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

SEQUENCE        = "room1"
N_FRAMES        = 24    # pool to subsample from
MAX_VGGT_FRAMES = 8     # hard cap per VGGT run — safe at 518 px on T4 16 GB
DOWNLOAD_DIR    = "/tmp/tumvi"

ds   = TUMVIDataset(sequence=SEQUENCE, n_frames=N_FRAMES, download_dir=DOWNLOAD_DIR)
ds.download()
data = ds.load()

image_paths      = data["image_paths"]
image_timestamps = data["image_timestamps"]
imu_readings     = data["imu_readings"]
gt_extrinsics    = data["gt_extrinsics"]   # (N, 3, 4)
calib            = data["calib"]

N      = len(image_paths)
span_s = image_timestamps[-1] - image_timestamps[0]
print(f"Pool : {N} frames over {span_s:.2f} s  ({N/span_s:.1f} fps effective)")
print(f"IMU  : {len(imu_readings)} readings  ({len(imu_readings)/span_s:.0f} Hz)")
print(f"Max frames per VGGT run: {MAX_VGGT_FRAMES}")

## 3 — Load Model

In [ ]:
pipe = VGGTPipeline()
pipe.load_model()
print(f"Model ready on {pipe.device}")

## 4 — IMU Frame Selection Visualisation

For each rotation threshold θ, show which frames from the 40-frame pool
are selected and what the cumulative rotation looks like across the sequence.

In [ ]:
gyro_bias = estimate_gyro_bias(imu_readings, duration_s=1.0)
print(f"Gyro bias: {gyro_bias} rad/s")

THETAS = [5, 8, 12, 18]   # degrees

# Section 4 shows uncapped selections for visualisation only.
# VGGT runs in Sections 5-6 use max_frames=MAX_VGGT_FRAMES.
imu_selections_viz = {}   # uncapped — for the plot
imu_selections     = {}   # capped   — for VGGT runs

for theta in THETAS:
    full = select_frames_by_rotation(
        imu_readings, image_timestamps,
        theta_min_deg=theta, gyro_bias=gyro_bias,
    )
    capped = select_frames_by_rotation(
        imu_readings, image_timestamps,
        theta_min_deg=theta, gyro_bias=gyro_bias,
        max_frames=MAX_VGGT_FRAMES,
    )
    imu_selections_viz[theta] = full
    imu_selections[theta]     = capped
    print(f"  θ={theta:>2}°  full={len(full):>2} frames  "
          f"capped={len(capped):>2} frames  (indices: {capped})")

In [ ]:
integrator = IMUPreintegrator(calibration=calib)
R_abs = integrator.gyro_only_rotations(imu_readings, image_timestamps, R0=np.eye(3))

from src.imu import so3_log
cum_rot_deg = [
    float(np.degrees(np.linalg.norm(so3_log(R_abs[i] @ R_abs[0].T))))
    for i in range(N)
]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=list(range(N)), y=cum_rot_deg,
    mode="lines", name="Cumulative rotation",
    line=dict(color="lightgrey", width=2),
))

colors = px.colors.qualitative.Plotly
for ci, theta in enumerate(THETAS):
    sel = imu_selections_viz[theta]   # uncapped for visualisation
    cap = imu_selections[theta]       # capped — mark differently
    fig.add_trace(go.Scatter(
        x=sel, y=[cum_rot_deg[i] for i in sel],
        mode="markers", name=f"θ={theta}° (full={len(sel)})",
        marker=dict(size=8, color=colors[ci], symbol="circle-open"),
    ))
    fig.add_trace(go.Scatter(
        x=cap, y=[cum_rot_deg[i] for i in cap],
        mode="markers", name=f"θ={theta}° (capped={len(cap)})",
        marker=dict(size=10, color=colors[ci], symbol="circle"),
        showlegend=False,
    ))

fig.add_hline(
    y=0, line_dash="dot", line_color="rgba(0,0,0,0)",
    annotation_text=f"Solid = capped at {MAX_VGGT_FRAMES} (fed to VGGT)  |  "
                    f"Open = full selection",
    annotation_position="bottom right",
)
fig.update_layout(
    title=f"IMU-guided frame selection — TUM-VI {SEQUENCE} (pool: {N} frames)",
    xaxis_title="Frame index in pool",
    yaxis_title="Cumulative rotation from frame 0 (°)",
    legend=dict(x=0.01, y=0.99),
)
fig.show()

## 5 — Accuracy vs N Frames: Uniform vs IMU-Guided (518 px)

For each θ value, we run VGGT on:
- **IMU-guided** selection (N frames, rotation-gated)
- **Uniform** selection (same N frames, evenly spaced)

Both at 518 px (native resolution).  Comparing at matched frame count
isolates the effect of frame selection quality from compute budget.

In [ ]:
os.makedirs("results", exist_ok=True)
rows = []

# Defensive: pull from whichever dict the selection cell created,
# then hard-slice to MAX_VGGT_FRAMES regardless.
_sel_source = locals().get("imu_selections", locals().get("imu_selections_viz", {}))

for theta in THETAS:
    imu_idx = _sel_source.get(theta, [])[:MAX_VGGT_FRAMES]
    N_sel   = len(imu_idx)
    if N_sel < 4:
        print(f"θ={theta}°: only {N_sel} frames after cap, skipping")
        continue

    # Uniform at same N, evenly spaced across the pool
    N_pool  = len(image_paths)
    uni_idx = np.round(np.linspace(0, N_pool - 1, N_sel)).astype(int).tolist()

    for method, indices in [("uniform", uni_idx), ("imu", imu_idx)]:
        paths_sel = [image_paths[i] for i in indices]
        gt_sel    = gt_extrinsics[np.array(indices)]

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        imgs, _ = load_images_from_list(paths_sel, target_size=518)

        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        t0  = time.perf_counter()
        out = run_vggt_inference(
            pipe.model, imgs, pipe.device, pipe.dtype, resolution=518
        )
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        elapsed = time.perf_counter() - t0
        peak_mb = (torch.cuda.max_memory_allocated() / 1024**2
                   if torch.cuda.is_available() else 0)

        ext = out["extrinsic"]
        ate = compute_ate(ext, gt_sel, align=True, with_scale=True)
        rpe = compute_rpe(ext, gt_sel, step=1)

        rows.append(dict(
            theta=theta, method=method, resolution=518,
            n_frames=N_sel, ate_mean=ate["mean"], ate_rmse=ate["rmse"],
            rpe_trans=rpe["trans_mean"], time_s=elapsed, peak_mb=peak_mb,
        ))
        print(f"θ={theta:>2}°  {method:<8}  518px  N={N_sel}  "
              f"ATE={ate['mean']:.4f}  t={elapsed:.1f}s  peak={peak_mb:.0f}MB")

        del imgs, out
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

df518 = pd.DataFrame(rows)
print("\nDone.")

In [ ]:
fig = go.Figure()

for method, color, dash in [("uniform", "royalblue", "solid"),
                             ("imu",     "firebrick",  "dash")]:
    sub = df518[df518["method"] == method].sort_values("n_frames")
    fig.add_trace(go.Scatter(
        x=sub["n_frames"], y=sub["ate_mean"],
        mode="lines+markers",
        name=f"{method} (518 px)",
        line=dict(color=color, width=2, dash=dash),
        marker=dict(size=9),
        text=[f"θ={t}°" for t in sub["theta"]],
        textposition="top center",
    ))

fig.update_layout(
    title="ATE vs N Frames — Uniform vs IMU-Guided Selection (518 px)",
    xaxis_title="Number of frames fed to VGGT",
    yaxis_title="ATE mean (m, Umeyama-aligned)",
    legend=dict(x=0.6, y=0.99),
)
fig.show()

## 6 — Combined Savings: IMU-Guided + 224 px

Phase 5 showed 224 px matches 518 px accuracy at 3× lower compute.
Here we combine both levers: IMU frame selection **and** 224 px input.

Baseline: uniform selection, 518 px (maximum compute).
Target  : IMU-guided selection, 224 px (minimum compute).

We compare ATE at each frame count to quantify the accuracy trade-off.

In [ ]:
rows_224 = []

_sel_source = locals().get("imu_selections", locals().get("imu_selections_viz", {}))

for theta in THETAS:
    imu_idx = _sel_source.get(theta, [])[:MAX_VGGT_FRAMES]
    N_sel   = len(imu_idx)
    if N_sel < 4:
        continue

    paths_sel = [image_paths[i] for i in imu_idx]
    gt_sel    = gt_extrinsics[np.array(imu_idx)]

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    imgs, _ = load_images_from_list(paths_sel, target_size=224)

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    t0  = time.perf_counter()
    out = run_vggt_inference(
        pipe.model, imgs, pipe.device, pipe.dtype, resolution=224
    )
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    peak_mb = (torch.cuda.max_memory_allocated() / 1024**2
               if torch.cuda.is_available() else 0)

    ext = out["extrinsic"]
    ate = compute_ate(ext, gt_sel, align=True, with_scale=True)
    rpe = compute_rpe(ext, gt_sel, step=1)

    rows_224.append(dict(
        theta=theta, method="imu", resolution=224,
        n_frames=N_sel, ate_mean=ate["mean"], ate_rmse=ate["rmse"],
        rpe_trans=rpe["trans_mean"], time_s=elapsed, peak_mb=peak_mb,
    ))
    print(f"θ={theta:>2}°  imu       224px  N={N_sel}  "
          f"ATE={ate['mean']:.4f}  t={elapsed:.1f}s  peak={peak_mb:.0f}MB")

    del imgs, out
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

df224 = pd.DataFrame(rows_224)
df_all = pd.concat([df518, df224], ignore_index=True)
print("\nDone.")

In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("ATE vs N Frames", "Inference Time vs N Frames"),
)

traces = [
    ("uniform", 518, "royalblue",  "solid",  "Uniform 518px (baseline)"),
    ("imu",     518, "darkorange", "dash",   "IMU-guided 518px"),
    ("imu",     224, "firebrick",  "dot",    "IMU-guided 224px (combined)"),
]

for method, res, color, dash, label in traces:
    sub = df_all[
        (df_all["method"] == method) & (df_all["resolution"] == res)
    ].sort_values("n_frames")
    if sub.empty:
        continue
    kwargs = dict(
        x=sub["n_frames"], mode="lines+markers", name=label,
        line=dict(color=color, width=2, dash=dash), marker=dict(size=9),
        legendgroup=label,
    )
    fig.add_trace(go.Scatter(y=sub["ate_mean"],  **kwargs), row=1, col=1)
    fig.add_trace(go.Scatter(y=sub["time_s"],
                             showlegend=False, **kwargs), row=1, col=2)

fig.update_xaxes(title_text="N frames")
fig.update_yaxes(title_text="ATE mean (m)",        row=1, col=1)
fig.update_yaxes(title_text="Inference time (s)",  row=1, col=2)
fig.update_layout(
    height=450,
    title_text="IMU frame selection + low resolution — combined compute savings",
    legend=dict(x=0.35, y=-0.2, orientation="h"),
)
fig.show()

## 7 — Summary Table

In [ ]:
# Reference: uniform 518px at the largest frame count in the sweep
ref = df518[(df518["method"] == "uniform")].sort_values("n_frames").iloc[-1]
ref_ate  = ref["ate_mean"]
ref_time = ref["time_s"]
ref_mb   = ref["peak_mb"]
ref_N    = int(ref["n_frames"])

summary_rows = []
for _, row in df_all.iterrows():
    summary_rows.append(dict(
        config    = f"{row['method']}-{int(row['resolution'])}px",
        theta_deg = row["theta"] if row["method"] == "imu" else "-",
        n_frames  = int(row["n_frames"]),
        ate_mean  = round(row["ate_mean"], 4),
        ate_delta = round(row["ate_mean"] - ref_ate, 4),
        time_s    = round(row["time_s"], 2),
        speedup_x = round(ref_time / row["time_s"], 2),
        peak_mb   = round(row["peak_mb"], 0),
        vram_saved= round(100 * (ref_mb - row["peak_mb"]) / ref_mb, 1)
                   if ref_mb > 0 else float("nan"),
    ))

df_summary = pd.DataFrame(summary_rows).sort_values(
    ["config", "n_frames"], ascending=[True, False]
)

print(f"Reference: uniform-518px  N={ref_N}  "
      f"ATE={ref_ate:.4f}m  time={ref_time:.1f}s  "
      f"peak={ref_mb:.0f}MB")
print()
print(df_summary.to_string(index=False, float_format="{:.4f}".format))

df_summary.to_csv("results/phase8_imu_frame_selection.csv", index=False)
print("\nSaved to results/phase8_imu_frame_selection.csv")

In [ ]:
# Bar chart: speedup and ATE delta for each config at its best theta
# (lowest ATE per config)
best = (
    df_summary.groupby("config")
    .apply(lambda g: g.loc[g["ate_mean"].idxmin()])
    .reset_index(drop=True)
)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Speedup vs baseline", "ATE delta vs baseline (m)"),
)

bar_colors = ["royalblue", "darkorange", "firebrick"]
for i, (_, row) in enumerate(best.iterrows()):
    label = f"{row['config']}\n(N={int(row['n_frames'])})"
    c = bar_colors[i % len(bar_colors)]
    fig.add_trace(
        go.Bar(name=label, x=[label], y=[row["speedup_x"]],
               marker_color=c, showlegend=False),
        row=1, col=1
    )
    fig.add_trace(
        go.Bar(name=label, x=[label], y=[row["ate_delta"]],
               marker_color=c, showlegend=False),
        row=1, col=2
    )

fig.add_hline(y=1.0, line_dash="dash", line_color="grey",
              annotation_text="1× (baseline)", row=1, col=1)
fig.add_hline(y=0.0, line_dash="dash", line_color="grey",
              annotation_text="0 Δ (baseline)", row=1, col=2)
fig.update_yaxes(title_text="Speedup (×)",   row=1, col=1)
fig.update_yaxes(title_text="ΔATE (m)",       row=1, col=2)
fig.update_layout(
    height=400,
    title_text="Best result per configuration vs uniform-518px baseline",
)
fig.show()

## 8 — Interpretation Guide

### What to look for

**ATE vs N frames plot (Section 5)**  
If the IMU-guided curve lies below the uniform curve at the same N, IMU
selection gives better pose accuracy for the same compute budget — it picks
frames with more rotational diversity, which is what VGGT needs.  If the
curves overlap, selection strategy does not matter for accuracy (still useful
because IMU selection can *reduce* N without accuracy loss).

**Combined savings plot (Section 6)**  
The key comparison is `imu-224px` vs `uniform-518px`.  If ΔATE is small
(< 10% relative), the combined strategy recovers the compute savings from
both Phase 5 (resolution) and Phase 8 (frame selection) without meaningful
accuracy loss.  The speedup column in the summary table quantifies this.

### Thesis framing

Phase 5 showed: **224 px = 518 px accuracy at 3× lower compute** (resolution lever).  
Phase 8 adds : **IMU-guided selection = uniform accuracy with fewer frames** (frame lever).  
Combined     : **IMU-guided + 224 px** addresses both axes of compute simultaneously,
targeting applications on edge devices (embedded VIO, AR headsets) where the
IMU is already available at zero marginal cost.

## 9 — Multi-Sequence Generalisation

Do the IMU frame selection benefits hold across different scene types?

| Sequence | Scene type | Motion |
|---|---|---|
| room1 | Indoor room (small) | Slow handheld pan |
| room2 | Indoor room (small) | Similar to room1 |
| corridor1 | Long corridor | Faster, rotational |
| slides1 | Near-planar lecture screen | Fast horizontal pan |

We run the same θ sweep for every sequence and compare:
- **uniform-518px** — baseline (current VGGT deployment)
- **imu-518px** — frame selection only (isolates selection quality)
- **imu-224px** — both levers combined (minimum compute)

The model is already loaded from Section 3 — we just swap the data.

In [ ]:
ALL_SEQUENCES = ["room1", "room2", "corridor1", "slides1"]
THETAS_MS     = [5, 8, 12, 18]   # same as single-sequence sweep
N_FRAMES_MS   = 24               # pool size per sequence
DOWNLOAD_DIR  = "/tmp/tumvi"
os.makedirs("results", exist_ok=True)

rows_ms = []

for seq in ALL_SEQUENCES:
    print(f"\n{'='*60}")
    print(f"Sequence: {seq}")
    print(f"{'='*60}")

    # --- Download / load ---
    ds_ms   = TUMVIDataset(sequence=seq, n_frames=N_FRAMES_MS,
                           download_dir=DOWNLOAD_DIR)
    ds_ms.download()
    data_ms = ds_ms.load()

    img_paths_ms  = data_ms["image_paths"]
    img_ts_ms     = data_ms["image_timestamps"]
    imu_ms        = data_ms["imu_readings"]
    gt_ms         = data_ms["gt_extrinsics"]
    calib_ms      = data_ms["calib"]
    N_ms          = len(img_paths_ms)
    span_ms       = img_ts_ms[-1] - img_ts_ms[0]
    print(f"  Pool: {N_ms} frames over {span_ms:.2f}s  |  "
          f"IMU: {len(imu_ms)} readings")

    # --- IMU bias + selections ---
    bias_ms = estimate_gyro_bias(imu_ms, duration_s=1.0)
    sels_ms = {}
    for theta in THETAS_MS:
        sels_ms[theta] = select_frames_by_rotation(
            imu_ms, img_ts_ms,
            theta_min_deg=theta, gyro_bias=bias_ms,
            max_frames=MAX_VGGT_FRAMES,
        )
        print(f"  θ={theta:>2}°  → {len(sels_ms[theta])} frames  "
              f"(indices: {sels_ms[theta]})")

    # --- Uniform baseline at each frame count used by IMU ---
    def _uniform_idx(n):
        return np.round(np.linspace(0, N_ms - 1, n)).astype(int).tolist()

    # --- Sweep: uniform-518, imu-518, imu-224 ---
    configs_ms = [
        ("uniform", 518),
        ("imu",     518),
        ("imu",     224),
    ]

    for theta in THETAS_MS:
        imu_idx = sels_ms[theta]
        N_sel   = len(imu_idx)
        if N_sel < 4:
            print(f"  θ={theta}°: only {N_sel} frames, skipping")
            continue
        uni_idx = _uniform_idx(N_sel)

        for method, res in configs_ms:
            indices = uni_idx if method == "uniform" else imu_idx
            paths_sel = [img_paths_ms[i] for i in indices]
            gt_sel    = gt_ms[np.array(indices)]

            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            imgs, _ = load_images_from_list(paths_sel, target_size=res)

            if torch.cuda.is_available():
                torch.cuda.reset_peak_memory_stats()
            t0  = time.perf_counter()
            out = run_vggt_inference(
                pipe.model, imgs, pipe.device, pipe.dtype, resolution=res
            )
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            elapsed = time.perf_counter() - t0
            peak_mb = (torch.cuda.max_memory_allocated() / 1024**2
                       if torch.cuda.is_available() else 0)

            ate = compute_ate(out["extrinsic"], gt_sel, align=True, with_scale=True)

            rows_ms.append(dict(
                sequence=seq,
                config=f"{method}-{res}px",
                theta_deg=theta if method == "imu" else None,
                n_frames=N_sel,
                ate_mean=round(ate["mean"], 4),
                time_s=round(elapsed, 2),
                peak_mb=round(peak_mb, 0),
            ))
            print(f"  θ={theta:>2}°  {method}-{res}px  "
                  f"N={N_sel}  ATE={ate['mean']:.4f}  "
                  f"t={elapsed:.1f}s  peak={peak_mb:.0f}MB")

            del imgs, out
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    del ds_ms, data_ms, imu_ms, gt_ms
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

df_ms = pd.DataFrame(rows_ms)

# Compute speedup and ΔATE relative to uniform-518px baseline per sequence
def _add_relative(df):
    out = []
    for seq, grp in df.groupby("sequence"):
        ref = grp[grp["config"] == "uniform-518px"]
        if ref.empty:
            out.append(grp)
            continue
        # Use per-theta uniform row (same N) as reference so comparison is fair
        for _, row in grp.iterrows():
            theta = row["theta_deg"]
            ref_row = ref[ref["n_frames"] == row["n_frames"]]
            if ref_row.empty:
                ref_row = ref.iloc[[0]]
            row = row.copy()
            row["ate_delta"]  = round(row["ate_mean"] - ref_row["ate_mean"].values[0], 4)
            row["speedup_x"]  = round(ref_row["time_s"].values[0] / row["time_s"], 2)
            row["vram_saved"] = round(
                100 * (ref_row["peak_mb"].values[0] - row["peak_mb"]) /
                max(ref_row["peak_mb"].values[0], 1), 1
            )
            out.append(row)
    return pd.DataFrame(out)

df_ms = _add_relative(df_ms)

df_ms.to_csv("results/phase8_multisequence.csv", index=False)
print("\n\nSaved to results/phase8_multisequence.csv")
print(df_ms[["sequence","config","theta_deg","n_frames",
             "ate_mean","ate_delta","speedup_x","vram_saved"]
            ].to_string(index=False, float_format="{:.3f}".format))

In [ ]:
# ── Multi-sequence comparison plots ─────────────────────────────────────────
# 1. ATE delta (vs per-sequence uniform-518px baseline) across θ, one subplot
#    per sequence.  Shows whether IMU selection helps/hurts relative to the
#    scene's own baseline.
# 2. Speedup bar chart: imu-224px at best-ATE θ per sequence.

SEQS_PLOT = ALL_SEQUENCES
fig_ate = make_subplots(
    rows=1, cols=len(SEQS_PLOT),
    subplot_titles=SEQS_PLOT,
    shared_yaxes=True,
)

cfg_style = {
    "imu-518px": ("darkorange", "dash",  "IMU-guided 518px"),
    "imu-224px": ("firebrick",  "solid", "IMU-guided 224px (combined)"),
}

for ci, seq in enumerate(SEQS_PLOT):
    sub = df_ms[df_ms["sequence"] == seq]
    for cfg, (color, dash, label) in cfg_style.items():
        rows_cfg = sub[sub["config"] == cfg].sort_values("theta_deg")
        if rows_cfg.empty:
            continue
        fig_ate.add_trace(go.Scatter(
            x=rows_cfg["theta_deg"],
            y=rows_cfg["ate_delta"],
            mode="lines+markers",
            name=label,
            legendgroup=label,
            showlegend=(ci == 0),
            line=dict(color=color, width=2, dash=dash),
            marker=dict(size=8),
        ), row=1, col=ci+1)

    fig_ate.add_hline(y=0, line_dash="dot", line_color="royalblue",
                      annotation_text="uniform-518px baseline",
                      annotation_position="top right",
                      row=1, col=ci+1)

fig_ate.update_xaxes(title_text="θ (°)")
fig_ate.update_yaxes(title_text="ΔATE vs baseline (m)", col=1)
fig_ate.update_layout(
    height=420,
    title_text="ATE delta vs uniform-518px baseline — across all sequences",
    legend=dict(x=0.3, y=-0.2, orientation="h"),
)
fig_ate.show()

# ── Speedup bar chart at best-ATE θ per sequence ───────────────────────────
best_ms = (
    df_ms[df_ms["config"] == "imu-224px"]
    .groupby("sequence")
    .apply(lambda g: g.loc[g["ate_mean"].idxmin()])
    .reset_index(drop=True)
)

fig_bar = make_subplots(rows=1, cols=3,
    subplot_titles=("Speedup (×)", "VRAM saved (%)", "ΔATE (m)"))

seq_colors = px.colors.qualitative.Plotly[:len(SEQS_PLOT)]
for i, (_, row) in enumerate(best_ms.iterrows()):
    c = seq_colors[i]
    lbl = f"{row['sequence']}\nθ={int(row['theta_deg'])}°"
    fig_bar.add_trace(go.Bar(x=[lbl], y=[row["speedup_x"]],
        marker_color=c, showlegend=False), row=1, col=1)
    fig_bar.add_trace(go.Bar(x=[lbl], y=[row["vram_saved"]],
        marker_color=c, showlegend=False), row=1, col=2)
    fig_bar.add_trace(go.Bar(x=[lbl], y=[row["ate_delta"]],
        marker_color=c, showlegend=False), row=1, col=3)

fig_bar.add_hline(y=1.0, line_dash="dash", line_color="grey", row=1, col=1)
fig_bar.add_hline(y=0.0, line_dash="dash", line_color="grey", row=1, col=3)
fig_bar.update_yaxes(title_text="Speedup (×)",    row=1, col=1)
fig_bar.update_yaxes(title_text="VRAM saved (%)", row=1, col=2)
fig_bar.update_yaxes(title_text="ΔATE (m)",       row=1, col=3)
fig_bar.update_layout(
    height=400,
    title_text="Best IMU-guided 224px result per sequence (vs own uniform-518px baseline)",
)
fig_bar.show()

print("\n=== Multi-sequence summary: best imu-224px per sequence ===")
print(best_ms[["sequence","theta_deg","n_frames","ate_mean",
               "ate_delta","speedup_x","vram_saved"]
              ].to_string(index=False, float_format="{:.3f}".format))